# Lec-1: Knowledge
---
**Init:** 
- Knowledge based reasoning.
- Sentence: assertion about the world in a knowledge representation
- Proposition Symbols:
  - P: It is raining,
  - Q: Harry visited Hagrid today, etc.

- Logical Connectives:
  - not: &not;
  - and: &and;
  - or: &or;
  - implication: -->
    - P --> Q : P implies Q.
    - _False_: only when P is True, but Q is False.
  - biconditional: <-->
    - P <--> Q : P if and only iff Q.
    - _True_: when both P and Q are False or both are True
    - _False_: when P and Q are in different True/False state.

- Model:
  - Assigns a truth value to every propositional word/phrase.

- Knowledge Base:
  - A set of sentences known by a knowledg-based agent.

- Entailment: &#8872;
  - $\alpha \models \beta$ : $\alpha$ entails $\beta$.
  - In every model (or world) in which sentence $\alpha$ is true, sentence $\beta$ is also true.

- **Inference:** 
> _Given Info:_
> - If it didn't rain, Harry visited Hagrid today.
> - Harry visited Hagrid or Dumbledore today, but not both.
> - Harry visited Dumbledore today.
> 
> _Machine Deduction or **Inference**_:
> - Harry did not visit Hagrid today.
> - It rained today.

### Example:
**Define Proposition Symbols:**
- **P**: _It is a Thursday._
- **Q**: _It is hot outside._
- **R**: _Harry will go for a run._

> Knowledge Base (KB): (P &and; &not; Q) $\rightarrow$ R.  
> Meaning: (It is a Thursday and It is not hot outside) implies/then Harry will go for a run.  
> Inference:  
> - P: True
> - &not; Q: Ture
> - (P &and; &not; Q): True
> - So, R: True. 

### Model Checking:
- To determine if $KB \models \alpha$,
  - Enumerate all possible models.
  - If in every model where KB is true, $\alpha$ is also true, then KB entails $\alpha$.
  - Otherwize, KB does not entail $\alpha$.

<img src="ModelChecking.png" alt="Model Checking" width="400"/>  

Let's see an example

In [2]:
from logic import *

rain = Symbol("rain") # it is raining.
hagrid = Symbol("hagrid") # Harry visited Hagrid.
dumbledore = Symbol("dumbledore") # Harry visited Dumbledore.

# Create Knowledge base (KB)
KB = And(
    Implication(Not(rain), hagrid), # If its not raining, Harry visited Hagrid,
    Or(hagrid, dumbledore), # Harry visited Hagrid or Dumbledore,
    Not(And(hagrid, dumbledore)), # Harry either visited Hagrid or Dumbledore, but not both,
    dumbledore # Harry visited dumbledore.
)

print("Knowledge Base (Machine Readable): \n", KB.formula())

# Model checking:
""" Given the KB, can we infer if 'it is raining today' ? """
print("Given the KB, can we infer if 'it is raining today': ", model_check(KB, rain))

Knowledge Base (Machine Readable): 
 ((¬rain) => hagrid) ∧ (hagrid ∨  dumbledore) ∧ (¬(hagrid ∧ dumbledore)) ∧ dumbledore
Given the KB, can we infer if 'it is raining today':  True


---

## Game : Clue
- Teach machine to play Clue.
- Based on the game rule of deduction.

**Propositional Symbols :**
| Person | Room | Weapon |
| ---    | ---  | ---    |
| mustard | ballroom | knife |
| plum | kitchen | revolver |
|scarlet | library | wrench |

**What we know:**
- Killer is one of these: (mustard &or; ballroom &or; knife )
- Killed in one of these: (ballroom &or; kitchen &or; library)
- Using on of these     : (knife &or; revolver &or; wrench)
- Player will pick card throughout the game and from those we can deduce more.

In [1]:
import termcolor
from logic import *

mustard    = Symbol("ColMustard") # Col. Mustard is the killer.
plum       = Symbol("ProfPlum")   # Prof. Plum is the killer.
scarlet    = Symbol("MsScarlet")  # Ms. Scarlet is the killer.
characters = [mustard, plum, scarlet]

ballroom = Symbol("Ballroom")  # Killed in Ballroom
kitchen  = Symbol("Kitchen")   # Killed in Kitchen
library  = Symbol("Library")   # Killed in Library
rooms    = [ballroom, kitchen, library]

knife    = Symbol("Knife")    # Using knife
revolver = Symbol("Revolver") # Using Revolver
wrench   = Symbol("Wrench")   # Using Wrench
weapons  = [knife, revolver, wrench]

symbols = characters + rooms + weapons

print("Symbol library: \n", symbols)


Symbol library: 
 [ColMustard, ProfPlum, MsScarlet, Ballroom, Kitchen, Library, Knife, Revolver, Wrench]


In [12]:
# [Func] Sequential Model Checking:
"""
First check if the symbol is true in the knowledge base (KB), i.e. if the card inside the envelope is Mustard, ballroom, and knife.
If it is true, print "YES" in green.
If it is not true, check if the negation of the symbol is true in the KB.
If the negation is not true, print "MAYBE". (because we don't know what is inside the envelope but it is not Mustard, ballroom, and knife)
"""
def check_knowledge(KB):
    for symbol in symbols:
        if model_check(KB, symbol):
            termcolor.cprint(f"{symbol}: YES", "green")
        elif not model_check(KB, Not(symbol)):
            termcolor.cprint(f"{symbol}: MAYBE", "yellow")

# Knowledge Base
KB = And(
    Or(mustard, plum, scarlet),
    Or(ballroom, kitchen, library),
    Or(knife, revolver, wrench)
)

print("Knoledge Base (Machine Readable): \n", KB.formula())

# Check the knowledge base
print("Checking the knowledge base...")
check_knowledge(KB)



Knoledge Base (Machine Readable): 
 (ColMustard ∨  ProfPlum ∨  MsScarlet) ∧ (Ballroom ∨  Kitchen ∨  Library) ∧ (Knife ∨  Revolver ∨  Wrench)
Checking the knowledge base...
ColMustard: MAYBE
ProfPlum: MAYBE
MsScarlet: MAYBE
Ballroom: MAYBE
Kitchen: MAYBE
Library: MAYBE
Knife: MAYBE
Revolver: MAYBE
Wrench: MAYBE


In [13]:
# Now lets add additional information to the knowledge base.
# 1st set of additional information:
KB.add(Not(mustard))  # Col. Mustard is not the killer.
KB.add(Not(ballroom))  # The murder did not take place in the ballroom.
KB.add(Not(revolver))  # The murder weapon was not the revolver.

# 2nd set of additional information:
KB.add(
    Or(Not(scarlet), Not(library), Not(wrench))
)# The murderer is either not Scarlet, or not in the Library, or not using the Wrench. One of these statements must be true, but not all

# 3rd set of additional information:
KB.add(Not(plum)) # Prof. Plum is not the killer. Somebody showed be the card.

# Check again
print("Checking the knowledge base...")
check_knowledge(KB)


Checking the knowledge base...
MsScarlet: YES
Kitchen: MAYBE
Library: MAYBE
Knife: MAYBE
Wrench: MAYBE


In [14]:
# 4th set of additional information:
KB.add(Not(library)) # murder did not take place in the library.
KB.add(Not(knife))   # knife is not the murder weapon.
# Check again
print("Checking the knowledge base...")
check_knowledge(KB)


Checking the knowledge base...
MsScarlet: YES
Kitchen: YES
Wrench: YES


# TODO: 
- [ ] Harry Potter 4 chars, 4 houses.

> Model Checking in not very efficient algorithm. As more and more variables are added, and logic complexity will make the algorithm less efficient.

---

# Inference Rules
**Clause:** _A Disjusction (connected with "or") or Conjunction (connected with "and") of literals._.  
_Empty clause is always FALSE!_


- **Modus Ponens:**
    - If known: $\alpha \rightarrow \beta$, and $\alpha$=True.
    - Then $\beta$ = True.

- **And Elimination:**
    - If known: $\alpha$ &and; $\beta$,
    - Then, $\alpha$ = True,
    - And, $\beta$ = True.

- **Double Negation Elimination:**
    - If known: &not;(&not;$\alpha$) is True,
    - Then, $\alpha$ = True.

- **Implication Elimination:**
    - If known: $\alpha \rightarrow \beta$,
    - Then, &not;$\alpha$ &or; $\beta$.   
    - Example: Known "If it is raining, Harry is inside", then "Either it is not raining or Harry is inside".

- **Biconditional Elimination:**
    - If known: $\alpha \leftrightarrow \beta$,
    - Then, ($\alpha \rightarrow \beta$) &and; (\beta \rightarrow \alpha)

- **De Morgan's Law:** _(turn "and" into "or", vice versa)_
    - If known: &not; ($\alpha$ &and; $\beta$),
    - Then, &not;$\alpha$ &or; &not;$\beta$.
    - Example: "It is not true that both Harry and Ron passed the test.", then,
        - Harry did not pass the test or Ron did not pass the test. 

- **Distributive Law:**

- **Unit Resolution Rule:**
    - If known:  ($P$ &or; $Q$), and &not;$P$ is True,
    - Then, $Q$ = True.
    - Generalization:
        - If known: ($P$ &or; $Q$), and (&not;$P$ &or; $R$),
        - Then, ($Q$ &or; $R$) = True.
        - Known:
            - "(Ron is in Great Hall) &or; (Hermione is in the Library)".
            - "(Ron is not in the Great Hall) &or; (Harry is sleeping)".
        - Inference:
            - "(Hermione is in the Library) &or; (Harry is sleeping)".

### Usecase: Theorem Proving.

- Init State: starting knowledge base,
- Actions: inference rules.
- Transition Model: new knowledge base after inference.
- Goal Test: Check statements we are trying to prove.
- Path cost function: number of steps in proof.

## Conjunctive Normal Form (CNF): 
- Sentences connected with "and" where each individual pieces has "or" in it.
- We usually turn any complex sentences into CNF.

**How to Convert to CNF ?** Using the inference rules. 
- Eliminate biconditionals
- Eliminate implications.
- Move &not; inwards using De Morgan's laws.
- Use Distributive law to distribute "and" and "or".

**Example:**    
~ (P &or; Q) $\rightarrow$ R    
~ &not;(P &or; Q) &or; R   (eliminate implication).   
~ (&not;P &and; &not;Q) &or; R   (De Morgan's Law).   
~ (&not;P &or; R) &and; (&not;P &or; R)   (distributive law). [Conjusction Normal Form]

### Inference by Resolution;

- To determine if KB $\models$ $\alpha$:
    - Check if (KB &and; &not;$\alpha$) is a contradiction ?
        - If so, then KB $\models$ $\alpha$.
        - Otherwise, no entailment.
    - We usually do this by converting "(KB &and; &not;$\alpha$)" to CNF, which are bunch of clauses added together.
    - Then we keep checking to see if we can use resolution to produce a new clause.
        - _If ever we produce the empty clause (equivalent of False), then we have a contradiction, then $\models$ $\alpha$._
        - _If we don't produce empty clause, then there is not entailment._

# TODO. 
- [ ] Example


### First Order Logic

- Constant Symbols
    - people, houses etc.
- Predicate Symbols
    - properties might hold True or False.
    - For example "Person" is True for "Minerva" and "Harry", but False for "Grifindor".
    - `Person(Minerva)`, Minerva is a person.
    - &not;`House(Minerva)`, Minerva is not a house.
    - `BelongsTo(Minerva, Gryffindor)`, Minerva belongs to Gryffindor.
    - Universal Quantification:
        - $\forall x$, `BelongsTo(x, Gryffindor)` $\rightarrow$ &not;`BelongsTo(x, Hufflepuff)`
    - Existantial Quantification:
        - $\exists x$, `House(x)` &and; `BelongsTo(Minerva, x)`.